# Neural task and distribution checks

This is an interface-validation report, not a model-selection benchmark.
Small training budgets exercise public methods across the registry. Metrics
are labeled by task and evaluation split; poor scores are not teaching targets.
For fitted teaching examples, use the [tutorials](../../tutorials/index.rst).


## Evidence at a glance

| Evidence | Scope of this report | Remaining limit |
|---|---|---|
| Regression and classification | Small fits across compatible registry entries; output and reconstruction checks | Training-row scores are not held-out performance estimates |
| Distributional and quantile objectives | Parameter domains, task-appropriate metrics, and component accounting | Finite outputs and valid domains do not establish calibration |
| Specialized neural behavior | See the [focused capability tests](../../architecture.rst) | Higher-order, sparsity, and local-importance checks are not exercised in this matrix |
| Warning evidence | Warnings captured per fit, including category, message and model/task | No warnings are classified as expected without an explicit rule |

Detailed tables and execution metadata follow. Registry support and a passed
check are separate claims.


## 1. Data and model setup

The same synthetic covariates are used throughout so differences in the
returned predictions, scores, and additive components reflect the modeled
quantity rather than an unrelated change of dataset.


In [1]:
from nampy.neural.configuration import FixedDesignTrainingConfig, IGANNTrainingConfig, ObjectiveConfig
from dataclasses import replace
from nampy.neural.configuration import GradientTrainingConfig, SemiStructuredTrainingConfig
from pathlib import Path
import tempfile
import time
import logging
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)

logging.getLogger("lightning.pytorch").setLevel(logging.ERROR)
plt.style.use("seaborn-v0_8-whitegrid")
COLORS = ["#2563EB", "#F97316", "#10B981", "#8B5CF6", "#EF4444"]

N_SAMPLES = 1000
MAX_EPOCHS = 20
RANDOM_STATE = 7
CHECKPOINT_ROOT = Path(tempfile.mkdtemp(prefix="nampy-notebook-"))

NEURAL_TRAINING = GradientTrainingConfig(max_epochs=MAX_EPOCHS, batch_size=256, random_state=RANDOM_STATE, checkpoint_path=CHECKPOINT_ROOT, trainer_kwargs={'logger': False, 'enable_progress_bar': False, 'enable_model_summary': False, 'num_sanity_val_steps': 0})

rng = np.random.default_rng(RANDOM_STATE)

from sklearn.metrics import accuracy_score, log_loss, roc_auc_score
from nampy import models
from nampy.neural.registry import architectures

n = N_SAMPLES
x = np.linspace(0.03, 0.97, n)
groups = pd.Categorical(np.resize(np.array(["a", "b", "c"]), n))
X = pd.DataFrame({"x1": x, "x2": np.cos(2 * np.pi * x), "group": groups})
y_reg = np.sin(2 * np.pi * x) + 0.25 * x + 0.2 * (groups == "b")
y_cls = np.resize(np.array([0, 1]), n)
y_lss = y_reg + np.exp(-1.2 + 0.8 * x) * rng.normal(size=n)

SWEEP_TRAINING = replace(NEURAL_TRAINING, batch_size=256, shuffle=False)

ARCHITECTURE_OPTIONS = {
    "linreg": {},
    "semistructured": {"structured_formula": "~ s(x1, k=4) + group",
                       "neural_features": ("x2",), "hidden_dims": (8,),
                       "representation_dim": 4},
    "nam": {"layer_sizes": [8], "dropout": 0.0},
    "sian": {"layer_sizes": [8], "interactions": [("x1", "x2")]},
    "snam": {"layer_sizes": [8], "dropout": 0.0, "group_lasso_lambda": 0.01},
    "gpnam": {"rff_num_feat": 8},
    "igann": {"n_hid": 4, "n_estimators": 3},
    "nbm": {"layer_sizes": [8], "num_bases": 4, "dropout": 0.0, "bases_dropout": 0.0, "output_penalty": 0.01},
    "nbm_spam": {"layer_sizes": [8], "num_bases": 4, "ranks": [4], "batch_norm": False},
    "spam": {"ranks": [4], "dropout": 0.0},
    "natt": {"d_model": 8, "n_layers": 1, "n_heads": 2, "transformer_dim_feedforward": 16, "head_layer_sizes": (), "attn_dropout": 0.0},
    "namformer": {"d_model": 8, "n_layers": 1, "n_heads": 2, "transformer_dim_feedforward": 16, "head_layer_sizes": (), "attn_dropout": 0.0},
    "treenam": {"tree_depth": 2, "tree_lamda": 0.01},
    "ensemble_treenam": {"tree_depth": 2, "tree_lamda": 0.01, "num_estimators": 2},
    "nodegam": {"num_trees": 4, "num_layers": 1, "depth": 2, "last_dropout": 0.0, "anneal_steps": 10, "interaction_degree": 1},
    "qnam": {"layer_sizes": [8], "dropout": 0.0},
    "spline_nam": {"n_knots": 5, "smoothing": 0.01},
}

def estimator_class(spec, suffix):
    return getattr(models, f"{spec.estimator_prefix}{suffix}", None)

def make_estimator(name, spec, suffix, *, family=None):
    cls = estimator_class(spec, suffix)
    if cls is None:
        return None
    training = SWEEP_TRAINING
    if name == "semistructured":
        training = SemiStructuredTrainingConfig(epochs=MAX_EPOCHS, random_state=RANDOM_STATE,
                                                shuffle=False, batch_size=256)
    elif name == "gpnam" and suffix == "Regressor":
        training = FixedDesignTrainingConfig(random_state=RANDOM_STATE, shuffle=False, batch_size=256)
    elif name == "igann" and suffix in {"Regressor", "Classifier"}:
        training = IGANNTrainingConfig(early_stopping=2, random_state=RANDOM_STATE, shuffle=False, batch_size=256)
    objective = None
    if suffix == "LSS":
        objective = ObjectiveConfig(
            family=family or spec.default_family,
            distributional_kwargs={"quantiles": [0.1, 0.5, 0.9]} if name == "qnam" else None,
        )
    return cls(architecture_config=spec.config(**ARCHITECTURE_OPTIONS[name]),
               objective=objective, training=training)

def reconstruction_error(estimator, features):
    components = estimator.predict_components(features, center=True)
    link = np.asarray(components.link)
    def aligned(value):
        array = np.asarray(value, dtype=float)
        if link.ndim == 1 and array.shape == (link.shape[0], 1):
            return array[:, 0]
        if link.ndim == 2 and array.shape == (link.shape[0],):
            return array[:, None]
        return array
    reconstruction = aligned(components.intercept)
    for contribution in components.terms.values():
        reconstruction = reconstruction + aligned(contribution)
    if components.offset is not None:
        reconstruction = reconstruction + aligned(components.offset)
    error = float(np.max(np.abs(reconstruction - link)))
    valid = error <= 1e-5
    return error, valid

warning_rows = []
def checked_fit(estimator, *args, label=None, **kwargs):
    # Capture every warning locally; unknown warnings remain visible evidence.
    with warnings.catch_warnings(record=True) as captured:
        warnings.simplefilter("always")
        try:
            return estimator.fit(*args, **kwargs)
        finally:
            warning_rows.extend({
                "model/task": label or type(estimator).__name__,
                "category": item.category.__name__, "message": str(item.message),
                "expected": False,
            } for item in captured)

def outcome(checks):
    return "Passed" if checks and all(bool(value) for value in checks.values()) else "Failed"

def array_checks(values, shape):
    values = np.asarray(values)
    return {"expected shape": values.shape == shape,
            "finite output": bool(np.isfinite(values).all())}

assert set(ARCHITECTURE_OPTIONS) == set(architectures()), "Architecture report configuration needs updating."


## 2. Training-data regression checks

Every registered regression architecture is fitted on the same rows. The
score, runtime, additive reconstruction, and term importance are recorded.
Architecture-specific output is added where available. This confirms the
installed runtime rather than relying on a copied documentation matrix.

Acceptance conditions and observed outcomes are recorded in the code and tables below.
These small fits do not test calibration or convergence. See the
[distributional guide](../../user_guide/gamlss.rst) for modeling context.


In [2]:
regression_models = {}
regression_rows = []
for name, spec in architectures().items():
    if not spec.supports("regression"):
        continue
    estimator = make_estimator(name, spec, "Regressor")
    started = time.perf_counter()
    checked_fit(estimator, X, y_reg, label=f'{name}/regression')
    importance = estimator.term_importance(X, center=True)
    error, reconstruction_valid = reconstruction_error(estimator, X)
    prediction = np.asarray(estimator.predict(X))
    mse = float(np.mean((prediction - y_reg)**2))
    checks = {**array_checks(prediction, (n,)),
              "reconstruction valid": reconstruction_valid, "finite MSE": np.isfinite(mse)}
    row = {
        "architecture": name,
        "status": outcome(checks), **checks, "MSE": mse,
        "R2": estimator.score(X, y_reg),
        "seconds": time.perf_counter() - started,
        "reconstruction error": error,
        "reconstruction valid": reconstruction_valid,
        "most important term": importance.iloc[0]["term"] if len(importance) else None,
    }
    if hasattr(estimator, "local_term_importance") and spec.supports("local_term_importance"):
        row["special output"] = "local importance"
    elif name == "igann":
        row["special output"] = f"{estimator.n_estimators_} boosting stages"
    else:
        row["special output"] = None
    regression_models[name] = estimator
    regression_rows.append(row)

regression_report = pd.DataFrame(regression_rows)
regression_report


,architecture,status,expected shape,finite output,reconstruction valid,finite MSE,MSE,R2,seconds,reconstruction error,most important term,special output
0,linreg,Passed,True,True,True,True,0.653105,-0.416730,1.545390,1.131147e-07,group,NaN
1,nam,Passed,True,True,True,True,0.785396,-0.703698,0.748947,2.314523e-08,x1,NaN
2,sian,Passed,True,True,True,True,0.454929,0.013159,0.608702,4.714727e-08,x1:x2,NaN
3,snam,Passed,True,True,True,True,0.608366,-0.319681,0.698615,2.618879e-08,group,NaN
4,semistructured,Passed,True,True,True,True,0.159883,0.653179,0.546992,2.220446e-16,"structured:s(x1, k=4)",NaN
5,gpnam,Passed,True,True,True,True,0.449705,0.024490,0.077166,2.579509e-08,group[1],NaN
6,igann,Passed,True,True,True,True,0.311275,0.324776,0.095062,3.803329e-08,group,3 boosting stages
7,nbm,Passed,True,True,True,True,0.504789,-0.094998,0.715970,1.301523e-08,group[1],NaN
8,nbm_spam,Passed,True,True,True,True,0.748337,-0.623309,0.933167,4.500058e-08,x1,NaN
9,spam,Passed,True,True,True,True,0.695666,-0.509053,0.626383,4.697098e-08,x1,local importance


## 3. Training-data classification checks

Classifier components reconstruct logits before the probability transform.
This matrix checks binary classification; multiclass behavior has separate
focused tests. Log loss evaluates probability quality,
ROC-AUC evaluates ranking for the positive class, accuracy evaluates hard
labels, and calibration error compares binned predicted and empirical
frequencies. Link-scale terms explain the model score—not an additive
decomposition of probability.

Acceptance conditions and observed outcomes are recorded in the code and tables below.
These small fits do not test calibration or convergence. See the
[distributional guide](../../user_guide/gamlss.rst) for modeling context.


In [3]:
def expected_calibration_error(y_true, probability, bins=6):
    edges = np.linspace(0, 1, bins + 1)
    index = np.clip(np.digitize(probability, edges) - 1, 0, bins - 1)
    total = len(y_true)
    return sum(
        np.sum(index == b) / total * abs(np.mean(probability[index == b]) - np.mean(y_true[index == b]))
        for b in range(bins) if np.any(index == b)
    )

classification_models = {}
classification_rows = []
for name, spec in architectures().items():
    if not spec.supports("classification"):
        continue
    estimator = make_estimator(name, spec, "Classifier")
    started = time.perf_counter()
    checked_fit(estimator, X, y_cls, label=f'{name}/classification')
    probabilities = np.asarray(estimator.predict_proba(X))
    probability = probabilities[:, 1]
    label = estimator.predict(X)
    importance = estimator.term_importance(X, center=True)
    error, reconstruction_valid = reconstruction_error(estimator, X)
    loss = log_loss(y_cls, probabilities)
    checks = {**array_checks(probabilities, (n, 2)),
              "valid probabilities": bool(((probabilities >= 0) & (probabilities <= 1)).all()),
              "normalized": bool(np.allclose(probabilities.sum(axis=1), 1, atol=1e-5)),
              "label shape": np.asarray(label).shape == (n,),
              "reconstruction valid": reconstruction_valid, "finite log loss": np.isfinite(loss)}
    classification_rows.append({
        "architecture": name, "status": outcome(checks), **checks,
        "log loss": log_loss(y_cls, probability),
        "ROC-AUC": roc_auc_score(y_cls, probability),
        "accuracy": accuracy_score(y_cls, label),
        "calibration error": expected_calibration_error(y_cls, probability),
        "reconstruction error": error,
        "reconstruction valid": reconstruction_valid,
        "most important term": importance.iloc[0]["term"] if len(importance) else None,
        "seconds": time.perf_counter() - started,
    })
    classification_models[name] = estimator
classification_report = pd.DataFrame(classification_rows)
classification_report


,architecture,status,expected shape,finite output,valid probabilities,normalized,label shape,reconstruction valid,finite log loss,log loss,ROC-AUC,accuracy,calibration error,reconstruction error,most important term,seconds
0,linreg,Passed,True,True,True,True,True,True,True,0.720785,0.500800,0.500,0.096465,1.554936e-07,group,1.046186
1,nam,Passed,True,True,True,True,True,True,True,0.703513,0.500876,0.500,0.065429,3.238767e-08,x1,0.869877
2,sian,Passed,True,True,True,True,True,True,True,0.694506,0.498844,0.498,0.022090,3.834814e-08,x1:x2,0.684026
3,snam,Passed,True,True,True,True,True,True,True,0.703215,0.500716,0.499,0.063022,2.700090e-08,group,0.944815
4,semistructured,Passed,True,True,True,True,True,True,True,0.693182,0.501156,0.500,0.004705,1.387779e-17,"structured:s(x1, k=4)",0.276512
5,gpnam,Passed,True,True,True,True,True,True,True,0.693147,0.500266,0.502,0.001803,2.031593e-10,group[0],0.822845
6,igann,Passed,True,True,True,True,True,True,True,0.693474,0.501000,0.501,0.011553,3.688037e-09,group,0.102213
7,nbm,Passed,True,True,True,True,True,True,True,0.696985,0.499374,0.500,0.037895,2.281251e-08,group[1],1.002980
8,nbm_spam,Passed,True,True,True,True,True,True,True,0.708713,0.499348,0.500,0.087498,3.641285e-08,x1,1.155993
9,spam,Passed,True,True,True,True,True,True,True,0.696847,0.501440,0.500,0.032660,4.007532e-08,x1,0.864013


## 4. Training-data location-scale checks

Check normal parameter and raw predictor shapes `(n, 2)`, finite outputs,
positive scales, finite NLL, and additive reconstruction (absolute tolerance
$10^{-5}$). Each compatible architecture uses the configuration above.
QNAM's quantile objective is checked separately.

The displayed intervals are $[\widehat\mu-1.645\widehat\sigma,\;
\widehat\mu+1.645\widehat\sigma]$, nominal 90% normal prediction intervals.
**Training-row coverage of nominal 90% intervals** describes fitting rows;
it is not evidence of held-out calibration. See the
[quantile calibration tutorial](../../in_depth/quantile_calibration.rst)
for calibration assessment.


In [4]:
lss_models = {}
lss_rows = []
for name, spec in architectures().items():
    if not spec.supports("distributional") or name == "qnam":
        continue
    estimator = make_estimator(name, spec, "LSS", family="normal")
    started = time.perf_counter()
    checked_fit(estimator, X, y_lss, label=f'{name}/normal LSS')
    parameters = estimator.predict(X)
    raw = estimator.predict(X, raw=True)
    lower = parameters[:, 0] - 1.645 * parameters[:, 1]
    upper = parameters[:, 0] + 1.645 * parameters[:, 1]
    error, reconstruction_valid = reconstruction_error(estimator, X)
    loss = estimator.evaluate(X, y_lss)["NLL"]
    checks = {**array_checks(parameters, (n, 2)),
              "raw shape": raw.shape == (n, 2), "finite raw": np.isfinite(raw).all(),
              "positive scale": bool((parameters[:, 1] > 0).all()),
              "finite NLL": np.isfinite(loss), "reconstruction valid": reconstruction_valid}
    lss_rows.append({
        "architecture": name, "status": outcome(checks), **checks,
        "NLL": estimator.evaluate(X, y_lss)["NLL"],
        "minimum scale": parameters[:, 1].min(),
        "Training-row coverage of nominal 90% intervals": np.mean((y_lss >= lower) & (y_lss <= upper)),
        "raw width": raw.shape[1],
        "reconstruction error": error,
        "reconstruction valid": reconstruction_valid,
        "seconds": time.perf_counter() - started,
    })
    lss_models[name] = estimator
lss_report = pd.DataFrame(lss_rows)
lss_report


,architecture,status,expected shape,finite output,raw shape,finite raw,positive scale,finite NLL,reconstruction valid,NLL,minimum scale,Training-row coverage of nominal 90% intervals,raw width,reconstruction error,seconds
0,linreg,Passed,True,True,True,True,True,True,True,1.298374,0.491920,0.916,2,1.227260e-07,0.764537
1,nam,Passed,True,True,True,True,True,True,True,1.351217,0.526378,0.784,2,3.762543e-08,0.666060
2,sian,Passed,True,True,True,True,True,True,True,1.182084,0.703069,0.957,2,6.476045e-08,0.811784
3,snam,Passed,True,True,True,True,True,True,True,1.248810,0.787541,0.921,2,5.653873e-08,0.771402
4,semistructured,Passed,True,True,True,True,True,True,True,0.917984,0.533968,0.951,2,2.220446e-16,0.288549
5,gpnam,Passed,True,True,True,True,True,True,True,1.219832,0.704296,0.856,2,7.947921e-09,0.653206
6,igann,Passed,True,True,True,True,True,True,True,1.234980,0.697946,0.839,2,2.143002e-09,0.662158
7,nbm,Passed,True,True,True,True,True,True,True,1.222408,0.682918,0.868,2,2.685800e-08,0.758688
8,nbm_spam,Passed,True,True,True,True,True,True,True,1.370343,0.540413,0.706,2,4.619174e-08,0.961055
9,spam,Passed,True,True,True,True,True,True,True,1.472334,0.531444,0.718,2,8.011441e-08,0.644846


## 5. Distribution families

A distribution family owns target support, output width, parameter
transforms, likelihood, point prediction, and default metrics. Continuous,
count, discrete/ordered, multivariate, and quantile families therefore
require different targets. The loop fits one small canonical NAMLSS model
per installed family. These fits test the interface and output domains;
they are not estimates worth interpreting scientifically.

Acceptance conditions and observed outcomes are recorded in the code and tables below.
These small fits do not test calibration or convergence. See the
[distributional guide](../../user_guide/gamlss.rst) for modeling context.

Ordinal outputs check `(n, C)` probabilities and the `(n, 1)` latent link.
BEINF similarly has five natural parameters and four raw predictors. Component
reconstruction is checked in raw predictor coordinates for both. Censored
families receive encoded observation intervals; beta-binomial trials are fixed
explicitly. Domain checks use the registered shared law where one is available.


In [5]:
from nampy.neural.configs.nam_config import DefaultNAMConfig

from nampy.neural.configuration import ObjectiveConfig
from nampy.models import NAMLSS
from nampy.neural.families.registry import FAMILY_REGISTRY
from nampy.distributions import ObservationTarget

positive = np.exp(0.2 + 0.4 * x + rng.normal(0, 0.15, n))
counts = rng.poisson(np.exp(0.2 + 0.5 * x))
zero_counts = counts.copy(); zero_counts[::3] = 0
classes3 = np.resize(np.arange(3), n)
composition = rng.dirichlet([2.0, 3.0, 4.0], size=n)
multi_y = np.column_stack([y_lss, np.cos(np.pi * x) + rng.normal(0, 0.1, n)])

beta_with_atoms = rng.beta(2, 3, n)
beta_with_atoms[::5] = 0
beta_with_atoms[1::5] = 1
positive_with_zeros = positive.copy()
positive_with_zeros[::3] = 0

def censored_target(values):
    lower, upper = values.copy(), values.copy()
    upper[::5] = np.inf
    lower[1::5] = -np.inf
    return ObservationTarget(lower, upper).encoded

family_targets = {
    "bccg": positive, "bct": positive, "bcpe": positive,
    "betabinomial": rng.binomial(10, 0.4, n),
    "beinf": beta_with_atoms, "zaga": positive_with_zeros,
    "normal_observation": censored_target(y_lss),
    "lognormal_observation": censored_target(positive),
    "weibull_observation": censored_target(positive),
    "normal": y_lss, "poisson": counts, "gamma": positive,
    "beta": np.clip(1 / (1 + np.exp(-y_lss)), 1e-4, 1-1e-4),
    "dirichlet": composition, "studentt": y_lss,
    "negativebinom": counts, "inversegamma": positive,
    "categorical": classes3, "quantile": y_lss, "robustnormal": y_lss,
    "lognormal": positive, "weibull": positive, "loglogistic": positive,
    "zip": zero_counts, "zinb": zero_counts,
    "hurdlepoisson": zero_counts, "hurdlenegativebinom": zero_counts,
    "tweedie": zero_counts.astype(float), "ordinal": classes3,
    "mvnormdiag": multi_y,
}
family_groups = {
    "continuous": {"bccg", "bct", "bcpe", "normal", "gamma", "beta", "studentt", "inversegamma", "robustnormal", "lognormal", "weibull", "loglogistic", "tweedie"},
    "counts": {"betabinomial", "poisson", "negativebinom", "zip", "zinb", "hurdlepoisson", "hurdlenegativebinom"},
    "discrete and ordered": {"categorical", "ordinal"},
    "multivariate": {"dirichlet", "mvnormdiag"},
    "quantiles": {"quantile"},
    "continuous with atoms": {"beinf", "zaga"},
    "censored continuous": {"normal_observation", "lognormal_observation", "weibull_observation"},
}

def group_for_family(name):
    return next(group for group, names in family_groups.items() if name in names)


def family_kwargs(name):
    if name == "quantile":
        return {"quantiles": [0.1, 0.5, 0.9]}
    if name == "betabinomial":
        return {"trials": 10}
    if name.endswith("_observation"):
        return {"lower": -5.0 if name == "normal_observation" else 0.01, "upper": 20.0}
    return {}


def family_checks(name, raw, natural, target):
    import torch

    distribution, _ = FAMILY_REGISTRY[name].instantiate(target, family_kwargs(name))
    width = distribution(torch.zeros((1, distribution.param_count))).shape[1]
    checks = {**array_checks(natural, (n, width)),
              "raw shape": raw.shape == (n, distribution.param_count),
              "finite raw": bool(np.isfinite(raw).all())}
    if natural.shape != (n, width):
        checks["valid domains"] = False
        return checks
    if name in {"categorical", "ordinal"}:
        checks["valid probabilities"] = bool(((natural >= 0) & (natural <= 1)).all())
        checks["normalized"] = bool(np.allclose(natural.sum(axis=1), 1, atol=1e-5))
    elif name == "quantile":
        checks["ordered quantiles"] = bool((np.diff(natural, axis=1) >= 0).all())
    elif FAMILY_REGISTRY[name].supports_distribution_prediction:
        from nampy.neural.families.distribution_adapter import distribution_prediction

        try:
            distribution_prediction(distribution, natural)
        except (ValueError, NotImplementedError):
            checks["valid domains"] = False
        else:
            checks["valid domains"] = True
    else:
        unrestricted = {"normal": {0}, "robustnormal": {0}, "studentt": {1},
                        "lognormal": {0}, "mvnormdiag": {0, 1}}.get(name, set())
        positive = [i for i in range(width) if i not in unrestricted]
        if name in {"zip", "zinb", "hurdlepoisson", "hurdlenegativebinom"}:
            positive.remove(0)
            checks["valid zero probability"] = bool(((natural[:, 0] > 0) & (natural[:, 0] < 1)).all())
        checks["valid domains"] = bool((natural[:, positive] > 0).all())
    return checks

assert set(family_targets) == set(FAMILY_REGISTRY), "Family report targets need updating."
assert set().union(*family_groups.values()) == set(FAMILY_REGISTRY), "Family report groups need updating."

family_models = {}
family_rows = []
for name in FAMILY_REGISTRY:
    estimator = NAMLSS(
        architecture_config=DefaultNAMConfig(layer_sizes=[8], dropout=0.0),
        objective=ObjectiveConfig(family=name,
            distributional_kwargs=family_kwargs(name)),
        training=SWEEP_TRAINING,
    )
    target = family_targets[name]
    started = time.perf_counter()
    checked_fit(estimator, X, target, label=f'nam/{name}')
    raw = estimator.predict(X, raw=True)
    natural = estimator.predict(X)
    metrics = estimator.evaluate(X, target)
    if name == "quantile":
        levels = np.asarray([0.1, 0.5, 0.9])
        residual = np.asarray(target)[:, None] - natural
        metric_name = "pinball loss"
        metric_value = float(np.maximum(levels * residual, (levels - 1) * residual).sum(axis=1).mean())
        if not (np.isclose(metric_value, metrics['pinball_loss'], rtol=1e-05)):
            raise ValueError("Validation failed: np.isclose(metric_value, metrics['pinball_loss'], rtol=1e-05)")
    else:
        metric_name, metric_value = "NLL", metrics["NLL"]
    # Components describe raw predictors independently of natural output width.
    error, reconstruction_valid = reconstruction_error(estimator, X)
    checks = {**family_checks(name, raw, natural, target),
              "finite loss": bool(np.isfinite(metric_value)),
              "reconstruction valid": reconstruction_valid}

    family_rows.append({
        "group": group_for_family(name), "family": name,
        "status": outcome(checks), **checks, "reconstruction error": error,
        "component extraction": outcome({"reconstruction": reconstruction_valid}), "target shape": np.asarray(target).shape,
        "raw parameters": raw.shape[1], "finite output": bool(np.isfinite(natural).all()),
        "metric": metric_name, "metric value": metric_value,
        "seconds": time.perf_counter() - started,
    })
    family_models[name] = estimator
family_report = pd.DataFrame(family_rows).sort_values(["group", "family"])
family_report


,group,family,status,expected shape,finite output,raw shape,finite raw,valid domains,finite loss,reconstruction valid,reconstruction error,component extraction,target shape,raw parameters,metric,metric value,seconds,valid probabilities,normalized,ordered quantiles,valid zero probability
2,censored continuous,lognormal_observation,Passed,True,True,True,True,True,True,True,5.552918e-08,Passed,"(1000, 3)",2,NLL,1.009000,0.775109,NaN,NaN,NaN,NaN
4,censored continuous,normal_observation,Passed,True,True,True,True,True,True,True,4.404783e-08,Passed,"(1000, 3)",2,NLL,1.137154,0.790707,NaN,NaN,NaN,NaN
3,censored continuous,weibull_observation,Passed,True,True,True,True,True,True,True,5.569309e-08,Passed,"(1000, 3)",2,NLL,1.185984,0.726864,NaN,NaN,NaN,NaN
5,continuous,bccg,Passed,True,True,True,True,True,True,True,6.524473e-08,Passed,"(1000,)",3,NLL,0.990807,0.685963,NaN,NaN,NaN,NaN
1,continuous,bcpe,Passed,True,True,True,True,True,True,True,8.060783e-08,Passed,"(1000,)",4,NLL,0.414786,4.215875,NaN,NaN,NaN,NaN
0,continuous,bct,Passed,True,True,True,True,True,True,True,7.706136e-08,Passed,"(1000,)",4,NLL,1.124871,3.674900,NaN,NaN,NaN,NaN
12,continuous,beta,Passed,True,True,True,True,True,True,True,3.840029e-08,Passed,"(1000,)",2,NLL,0.175786,0.682671,NaN,NaN,NaN,NaN
11,continuous,gamma,Passed,True,True,True,True,True,True,True,8.023530e-08,Passed,"(1000,)",2,NLL,1.575774,0.651036,NaN,NaN,NaN,NaN
16,continuous,inversegamma,Passed,True,True,True,True,True,True,True,4.896522e-08,Passed,"(1000,)",2,NLL,1.659127,0.731386,NaN,NaN,NaN,NaN
22,continuous,loglogistic,Passed,True,True,True,True,True,True,True,6.133318e-08,Passed,"(1000,)",2,NLL,2.220873,0.680834,NaN,NaN,NaN,NaN


### 5.1 Inspect parameter constraints

Natural transforms enforce positive rates/scales and unit-interval
probabilities. They do not guarantee calibration. QNAM additionally
enforces ordered quantiles numerically.

Acceptance conditions and observed outcomes are recorded in the code and tables below.
These small fits do not test calibration or convergence. See the
[distributional guide](../../user_guide/gamlss.rst) for modeling context.


In [6]:
domain_checks = {
    "normal scale positive": bool((family_models["normal"].predict(X)[:, 1] > 0).all()),
    "beta parameters positive": bool((family_models["beta"].predict(X) > 0).all()),
    "ZIP finite": bool(np.isfinite(family_models["zip"].predict(X)).all()),
}

qnam = make_estimator("qnam", architectures()["qnam"], "LSS")
checked_fit(qnam, X, y_lss, label='qnam/quantile')
quantiles = qnam.predict(X)
minimum_quantile_gap = float(np.diff(quantiles, axis=1).min())
domain_checks["QNAM minimum adjacent quantile gap"] = minimum_quantile_gap

error, reconstruction_valid = reconstruction_error(qnam, X)
q_loss = qnam.evaluate(X, y_lss)["pinball_loss"]
quantile_checks = {**array_checks(quantiles, (n, 3)),
                   "ordered quantiles": minimum_quantile_gap >= 0,
                   "reconstruction valid": reconstruction_valid, "finite loss": np.isfinite(q_loss)}
quantile_status = outcome(quantile_checks)
quantile_verified = quantile_status == "Passed"
display({"QNAM": quantile_status, **quantile_checks})
domain_checks


{'QNAM': 'Passed',
 'expected shape': True,
 'finite output': True,
 'ordered quantiles': True,
 'reconstruction valid': True,
 'finite loss': True}

{'normal scale positive': True,
 'beta parameters positive': True,
 'ZIP finite': True,
 'QNAM minimum adjacent quantile gap': 2.7959256172180176}

## 6. Ensemble uncertainty

`NeuralEnsemble` clones and fits regressors or classifiers independently.
Bootstrapping changes member datasets; random seeds change initialization
and data order; `n_jobs` controls parallel fitting. Predictions are
averaged on the response scale, while terms and links are averaged on the
additive scale. Persistence is inherited from the member estimators only;
the complete ensemble can be serialized with joblib.

Acceptance conditions and observed outcomes are recorded in the code and tables below.
These small fits do not test calibration or convergence. See the
[distributional guide](../../user_guide/gamlss.rst) for modeling context.


In [7]:

from nampy.neural.configs.nam_config import DefaultNAMConfig
from joblib import dump, load
from nampy.models import NAMRegressor, NeuralEnsemble

ensemble = NeuralEnsemble(NAMRegressor(architecture_config=DefaultNAMConfig(layer_sizes=[8], dropout=0.0)), n_estimators=3, random_state=RANDOM_STATE, n_jobs=1, bootstrap=True)
checked_fit(ensemble.set_params(estimator__training=SWEEP_TRAINING), X, y_reg, label='ensemble/regression')
ensemble_prediction = ensemble.predict(X)
ensemble_components = ensemble.predict_components(X, center=True)
uncertainty = ensemble.predict_component_uncertainty(X, center=True)
ensemble_reconstruction_error, _ = reconstruction_error(ensemble, X)

ensemble_path = CHECKPOINT_ROOT / "neural-ensemble.joblib"
dump(ensemble, ensemble_path)
restored_ensemble = load(ensemble_path)
reload_difference = float(np.max(np.abs(restored_ensemble.predict(X) - ensemble_prediction)))

display({
    "members": uncertainty.n_estimators,
    "mean response SD": float(np.mean(np.std([member.predict(X) for member in ensemble.estimators_], axis=0))),
    "mean term SD": {name: float(values.mean()) for name, values in uncertainty.term_std.items()},
    "reconstruction error": ensemble_reconstruction_error,
    "maximum reload difference": reload_difference,
})


{'members': 3,
 'mean response SD': 0.1983688920736313,
 'mean term SD': {'x1': 0.04003573446966001,
  'x2': 0.12111195366401764,
  'group': 0.034382466944564045},
 'reconstruction error': 4.280358556485453e-08,
 'maximum reload difference': 0.0}

### 6.1 Three different meanings of uncertainty

- **Ensemble disagreement** measures variation among fitted neural members
  under seeds and/or bootstrap samples. It is not automatically a
  calibrated confidence interval.
- **GAM standard errors** arise from an approximate sampling covariance for
  fitted coefficients, conditional on model assumptions and optionally
  including smoothing uncertainty.
- **LSS conditional uncertainty** is the predicted response distribution
  for a new outcome given its features. It can be large even if parameter
  estimates are known precisely.


## 7. Supported tasks and checked outputs

Outcomes are **Passed**, **Failed**, **Not run**, or **Unsupported**. Passed
requires every stated check, including shape, finite values, domains,
probability normalization where relevant, reconstruction (absolute tolerance
`1e-5`), and finite task loss. Scores are secondary training diagnostics.
Warnings are retained separately and do not establish convergence.

Evidence codes R, C, L and Q identify passed regression, classification,
normal LSS and QNAM checks. F identifies the canonical NAM quantile-family
check. Term evidence is extracted only from passed fits.

Quantile objective support through a compatible LSS estimator is separate
from the structural noncrossing guarantee demonstrated for QNAM. An unrun
combination supplies no evidence of unavailability. See the
[capability reference](../../models/index.rst) for estimator restrictions.


In [8]:
def task_status(name, capability, report):
    spec = architectures()[name]
    if not spec.supports(capability):
        return "Unsupported"
    match = report.loc[report["architecture"] == name]
    if len(match):
        return match.iloc[0]["status"]
    return "Not run"

def passed_evidence(name):
    codes = [label for label, capability, report in [
        ("R", "regression", regression_report),
        ("C", "classification", classification_report),
        ("L", "distributional", lss_report)]
        if task_status(name, capability, report) == "Passed"]
    if name == "qnam" and quantile_status == "Passed":
        codes.append("Q")
    if name == "nam" and family_report.loc[family_report.family == "quantile", "status"].eq("Passed").any():
        codes.append("F")
    return ", ".join(codes)

def demonstrated_term_kinds(name):
    fitted = [models[name] for models in [regression_models, classification_models, lss_models]
              if name in models and task_status(name, "distributional" if models is lss_models else "classification" if models is classification_models else "regression", lss_report if models is lss_models else classification_report if models is classification_models else regression_report) == "Passed"]
    if name == "qnam" and quantile_verified:
        fitted.append(qnam)
    observed = set()
    for estimator in fitted:
        components = estimator.predict_components(X)
        if not (np.isfinite(np.asarray(components.link)).all()):
            raise ValueError('Validation failed: np.isfinite(np.asarray(components.link)).all()')
        components.validate_additive_reconstruction(rtol=1e-5, atol=1e-5)
        for term, contribution in components.terms.items():
            values = np.asarray(contribution)
            if not (len(values) == len(X) and np.isfinite(values).all()):
                raise ValueError('Validation failed: len(values) == len(X) and np.isfinite(values).all()')
            metadata = components.term_metadata.get(term)
            if metadata is not None:
                observed.add(metadata.kind)
    return observed

capability_rows = []
for name, spec in architectures().items():
    observed = demonstrated_term_kinds(name)
    capability_rows.append({
        "model": name,
        "Task evidence": passed_evidence(name),
        "Regression": task_status(name, "regression", regression_report),
        "Classification": task_status(name, "classification", classification_report),
        "LSS": ("Passed" if quantile_verified else "Not run") if name == "qnam" else task_status(name, "distributional", lss_report),
        **{label: "Passed" if kind in observed else "Not returned in checked fits"
           for label, kind in [("Main effects", "main"), ("Contextual terms", "contextual"), ("Global terms", "global")]},
        "Interactions": "Passed" if "interaction" in observed else ("Not run" if spec.supports("interactions") else "Unsupported"),
        "Higher-order interactions": "Not run" if name in {"sian", "nbm", "spam", "nbm_spam"} else "Not applicable",
        "Sparsity": "Not run" if name in {"snam", "sian", "nbm", "igann"} else "Not applicable",
        "Local importance": "Not run" if spec.supports("local_term_importance") else "Unsupported",
        "Quantile objective": "Supported" if spec.supports("distributional") else "Unsupported",
        "Structural noncrossing quantiles": "Demonstrated" if name == "qnam" and quantile_verified else "Not assessed",
        "Quantile check in this report": quantile_status if name == "qnam" else (family_report.loc[family_report.family == "quantile", "status"].iloc[0] if name == "nam" else "Not run"),
    })

final_capability_report = pd.DataFrame(capability_rows)
display(final_capability_report.drop(columns=["Task evidence"]))
display(final_capability_report[["model", "Task evidence"]])

warning_report = pd.DataFrame(warning_rows, columns=["model/task", "category", "message", "expected"])
warning_summary = (warning_report.groupby(["model/task", "category", "message", "expected"],
                                          dropna=False).size().reset_index(name="count"))
display(warning_summary if len(warning_summary) else "No fit warnings captured.")
# Fail execution only after displaying all outcomes and warning evidence.
failed = any((report["status"] == "Failed").any() for report in
             [regression_report, classification_report, lss_report, family_report]) or not quantile_verified
if failed:
    raise AssertionError("Validation checks failed; inspect the outcome tables above.")


,model,Regression,Classification,LSS,Main effects,Contextual terms,Global terms,Interactions,Higher-order interactions,Sparsity,Local importance,Quantile objective,Structural noncrossing quantiles,Quantile check in this report
0,linreg,Passed,Passed,Passed,Passed,Not returned in checked fits,Not returned in checked fits,Unsupported,Not applicable,Not applicable,Unsupported,Supported,Not assessed,Not run
1,nam,Passed,Passed,Passed,Passed,Not returned in checked fits,Not returned in checked fits,Not run,Not applicable,Not applicable,Unsupported,Supported,Not assessed,Passed
2,sian,Passed,Passed,Passed,Passed,Not returned in checked fits,Not returned in checked fits,Passed,Not run,Not run,Unsupported,Supported,Not assessed,Not run
3,snam,Passed,Passed,Passed,Passed,Not returned in checked fits,Not returned in checked fits,Not run,Not applicable,Not run,Unsupported,Supported,Not assessed,Not run
4,semistructured,Passed,Passed,Passed,Passed,Not returned in checked fits,Passed,Unsupported,Not applicable,Not applicable,Unsupported,Supported,Not assessed,Not run
5,gpnam,Passed,Passed,Passed,Passed,Not returned in checked fits,Not returned in checked fits,Not run,Not applicable,Not applicable,Unsupported,Supported,Not assessed,Not run
6,igann,Passed,Passed,Passed,Passed,Not returned in checked fits,Not returned in checked fits,Unsupported,Not applicable,Not run,Unsupported,Supported,Not assessed,Not run
7,nbm,Passed,Passed,Passed,Passed,Not returned in checked fits,Not returned in checked fits,Not run,Not run,Not run,Unsupported,Supported,Not assessed,Not run
8,nbm_spam,Passed,Passed,Passed,Passed,Not returned in checked fits,Not returned in checked fits,Not run,Not run,Not applicable,Unsupported,Supported,Not assessed,Not run
9,spam,Passed,Passed,Passed,Passed,Not returned in checked fits,Not returned in checked fits,Not run,Not run,Not applicable,Not run,Supported,Not assessed,Not run


,model,Task evidence
0,linreg,"R, C, L"
1,nam,"R, C, L, F"
2,sian,"R, C, L"
3,snam,"R, C, L"
4,semistructured,"R, C, L"
5,gpnam,"R, C, L"
6,igann,"R, C, L"
7,nbm,"R, C, L"
8,nbm_spam,"R, C, L"
9,spam,"R, C, L"


,model/task,category,message,expected,count
0,ensemble/regression,FutureWarning,"`isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.",False,6
1,ensemble/regression,PossibleUserWarning,The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.,False,3
2,ensemble/regression,PossibleUserWarning,The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.,False,3
3,ensemble/regression,UserWarning,"You called `self.log('train_loss', ..., logger=True)` but have no logger configured. You can enable one by doing `Trainer(logger=ALogger(...))`",False,240
4,ensemble/regression,UserWarning,"You called `self.log('val_loss', ..., logger=True)` but have no logger configured. You can enable one by doing `Trainer(logger=ALogger(...))`",False,60
5,ensemble_treenam/classification,FutureWarning,"`isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.",False,2
6,ensemble_treenam/classification,PossibleUserWarning,The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.,False,1
7,ensemble_treenam/classification,PossibleUserWarning,The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.,False,1
8,ensemble_treenam/classification,UserWarning,"You called `self.log('train_acc', ..., logger=True)` but have no logger configured. You can enable one by doing `Trainer(logger=ALogger(...))`",False,80
9,ensemble_treenam/classification,UserWarning,"You called `self.log('train_auroc', ..., logger=True)` but have no logger configured. You can enable one by doing `Trainer(logger=ALogger(...))`",False,80


## 8. Interpreting the results

Successful execution verifies the estimator interface and output shapes;
it does not establish that an architecture is suitable for a particular
dataset. For a substantive comparison, choose metrics before fitting, tune
inside cross-validation, use repeated splits, inspect term stability, and
compare against simple baselines.
